# 🎤 super_Voz - StyleTTS2 Kaggle Pipeline

Este notebook executa o treinamento completo do StyleTTS2 no Kaggle.

### 🚀 Como usar:
1. Vá em **Settings** e ative a **GPU** (P100 ou T4 x2).
2. Certifique-se de que a **Internet** está ligada em **Settings**.
3. Clique em **Run All**.

### 📂 Dados no Kaggle:
O Kaggle **não permite montar pastas do Google Drive** como o Colab faz. Para usar seus áudios aqui, você tem duas opções:
1. **Kaggle Dataset:** Crie um Dataset no Kaggle com seus áudios e adicione-o a este notebook. O script procurará em `/kaggle/input/super-voz`.
2. **Google Drive (via gdown):** Compacte seus áudios em um arquivo `.zip`, faça o upload no Drive, deixe o link público e coloque o **ID do arquivo** no `styletts2_kaggle_config.yml`.

In [ ]:
# 🛠️ 1. Configurar Repositório e Ambiente
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/warllemedicao/Voz_styllett2.git"
PROJECT_ROOT = Path("/kaggle/working/Super_voz")

print("--- Preparando Repositório ---")
if PROJECT_ROOT.exists():
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull"], check=False)
else:
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)

# Localiza o subdiretório do projeto
PROJECT_DIR = PROJECT_ROOT / "super_Voz"
if not PROJECT_DIR.exists(): PROJECT_DIR = PROJECT_ROOT

os.chdir(PROJECT_DIR)
print(f"✅ Pronto! Diretório atual: {os.getcwd()}")

In [ ]:
# 📝 2. Criar Script de Pipeline (Bootstrap)
# Criamos o script aqui para garantir que ele exista mesmo que o Git esteja desatualizado.

pipeline_code = """
import argparse, os, shutil, subprocess, sys, yaml
from pathlib import Path

def run(cmd, cwd=None): 
    print("\\n$ " + " ".join(map(str, cmd)))
    subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=True)

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True)
    args = parser.parse_args()
    
    with open(args.config, 'r') as f: cfg = yaml.safe_load(f)
    
    # 1. Instalar dependencias basicas
    run([sys.executable, "-m", "pip", "install", "-q", "gdown", "accelerate", "huggingface_hub", "librosa", "soundfile", "phonemizer", "pyyaml"])
    run(["apt-get", "update"], check=False)
    run(["apt-get", "install", "-y", "ffmpeg", "sox", "espeak-ng"], check=False)
    
    # 2. Executar script real se existir, ou usar logica embutida simplificada
    script_real = Path("scripts/run_kaggle_styletts2.py")
    if not script_real.exists():
        # Tenta o script unificado
        script_real = Path("run_pipeline.py")
    
    if script_real.exists():
        print(f"--- Executando Pipeline: {script_real} ---")
        os.execv(sys.executable, [sys.executable, str(script_real), "--config", args.config])
    else:
        print("❌ ERRO CRÍTICO: Script de execução não encontrado no repositório!")
        print("Verifique se você deu 'git push' nos arquivos novos ou se a estrutura de pastas está correta.")
        sys.exit(1)

if __name__ == '__main__': main()
"""

with open("bootstrap_pipeline.py", "w") as f:
    f.write(pipeline_code.strip())
print("✅ Script de bootstrap criado.")

In [ ]:
# 🚀 3. Iniciar Pipeline
import sys
!{sys.executable} bootstrap_pipeline.py --config styletts2_kaggle_config.yml